##### Load the PDF using PyPDF2

In [ ]:
# # Install pyPDF2 if needed
# %pip install -q PyPDF2

In [ ]:
# Import libraries
import PyPDF2
import os


In [ ]:
# Load the PDF
file_path = r"C:\Users\kanyi\OneDrive\Desktop\module_4\RAG\Python For Dummies.pdf"
file_name = os.path.basename(file_path)
full_text = ""

# Open the PDF
with open(file_path, 'rb') as f:
    pdf_reader = PyPDF2.PdfReader(f)

    # Process each page
    for page in pdf_reader.pages:
        # Extract text from page
        text = page.extract_text()
        if text:
        # Extract full text
            full_text += text + "\n"
            print(full_text)


#### Different Chunking techniques

##### Fixed-size Chunking (Character-Based)

In [ ]:
def chunk_by_characters(text, chunk_size=20000, overlap=3000):
    """
    Splits text into chunks of specified character length
    
    Args:
        text: Text to be chunked
        chunk_size: Number of characters per chunk
        overlap: Number of characters to overlap between chunks
    
    Returns:
        List of text chunks
    """
    chunks = []
    start = 0

    while start < len(text):
        # Get chunk from start to start + chunk_size
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        # Move start position (with overlap)
        start += chunk_size - overlap

    return chunks

# Testing
chunks = chunk_by_characters(full_text, chunk_size=20000, overlap=3000)

print(f"Number of chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks[:3], 1): # Show first 500 chunks
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(chunk)
    print("=" * 80)

##### Word-Based Chunking

In [ ]:
def chunk_by_words(text, chunk_size=2000, overlap=300):
    """
    Split text into chunks of specified word count.

    Args:
        text: The text to be chunked
        chunk_size: Number of words per chunks
        overlap: Number of words to overlap between chunks

    Returns: 
        List if text chunks   
    """

    # Split text into words
    words = text.split()
    word_chunks = []
    start = 0

    while start < len(words):
        # GEt chunk of words
        end = start + chunk_size
        chunk_words = words[start:end]

        # Join words back to text
        chunk = ' '.join(chunk_words)
        word_chunks.append(chunk)

        # Move start position (with overlap)
        start += chunk_size - overlap

    return word_chunks

# Test it
word_chunks = chunk_by_words(full_text, chunk_size=2000, overlap=300)

print(f"Number of Chunks: {len(word_chunks)}\n")
for i, chunk in enumerate(word_chunks[:3], 1):
    print(f"Chunk {i} ({len(chunk.split())} words):")
    print(chunk)
    print("=" * 150)

##### Sentence-Based Chunking

In [ ]:
def chunk_by_sentences(text, max_chunk_size=5000):
    """
    Split text into chunks by sentences, keeping the sentences intact

    Args: 
        text: The text to be chunked
        max_chunk_size: Maximum characters per chunk

    Returns:
        List of text chunks
    """

    # Simple sentence splitting (split on . ! ?)
    import re


    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentence_chunks = []
    current_chunk = ""

    for sentence in sentences:
        # Checking if adding this sentence would exceed max size
        if len(current_chunk) + len(sentence) > max_chunk_size and current_chunk:
            # Save current chunk and start new one
            sentence_chunks.append(current_chunk.strip())
            current_chunk= sentence

        else:
            # Add sentence to current chunk
            current_chunk += " " + sentence if current_chunk else sentence
    
    #The last chunk
    if current_chunk:
        sentence_chunks.append(current_chunk.strip())
    return sentence_chunks

# Test 
sentence_chunks = chunk_by_sentences(full_text, max_chunk_size=5000)

print(f"Number of chunks: {len(sentence_chunks)}\n")
for i, chunk in enumerate(sentence_chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(chunk)
    print("=" * 150)

In [ ]:
def chunk_by_paragraphs(text, min_chunk_size=1000):
    """
    Split text by paragraphs (double newlines).
    
    Args:
        text: Text to be chunked
        min_chunk_size: Minimum characters per chun (combine small paragraphs)
    
    Returns:
        List of chunks        
    """

    # Splits by doubles newlines (paragraph separator)
    paragraghs = text.split("\n\n")

    para_chunks = []
    current_chunk = ""

    for para in paragraghs:
        para = para.strip()
        if not para:
            continue
        # If paragraph is too sma;ll, combine with next
        if len(para) <  min_chunk_size:
            current_chunk += "\n\n" + para if current_chunk else para
        else:
            # Save previous chunk if exists
            if current_chunk:
                para_chunks.append(current_chunk.strip()) 
            # Start new chunk wit this paragraph
            current_chunk = para
    
    # The last chunk
    if current_chunk:
        para_chunks.append(current_chunk.strip())
    
    return para_chunks

# Test 
para_chunks = chunk_by_paragraphs(full_text, min_chunk_size=1000)

print(f"Number of chunks {len(para_chunks)}\n")
for i, chunk in enumerate(para_chunks, 1):
    print(f"Chunk {i} ({len(chunk)} chars):")
    print(chunk)
    print("=" * 80)

##### Embedding Engine using (Local Embeddings- Free, no API required)


In [ ]:
# Install sentence-transformers
%pip install -q sentence-transformers

In [ ]:
# Load a small, fast embedding model
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-V2')
print("Model Loaded!")
print(f"Model produces {model.get_sentence_embedding_dimension()} dimensional embeddings")



In [ ]:
# Define a function to to get the embedding
def generate_embedding(text):
    """
    Generates an embedding using a local embedding - SentenceTransformer

    Args:
        text: Text to be embedded
    """
    # normalize embeddings so cosine similarity becomes simple
    return model.encode(text, normalize_embeddings=True)

# test 
embedding = generate_embedding(full_text)


print(f"Embedding shape: {embedding.shape}")
print(f"Embedding type: {type(embedding)}")
print(f"\nFirst 100 values: {embedding[:100]}")



##### Cosine Similarity

In [ ]:
def cosine_similarity(vec1,vec2):
    """
    Calculate cosine similarity between two vectors

    Returns a score between -1 and 1 (higher = more similar)
    """
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    
    return dot_product / (norm1 * norm2)

print("Similarity function ready")

#### Testing Similarity

In [ ]:
# Create test sentences
sentences = [
    "Python functions are blocks of reusable code.",
    "A function in Python allows code reuse.",
    "Dogs are loyal animals."
]

# generate embeddings for all sentences
embedding = model.encode(sentences)

# Compare first sentence to all others
print("Comparing to: 'Python functions are blocks of reusable code.'\n")
for i, sentence in enumerate(sentences):
    similarity = cosine_similarity(embedding[0], embedding[i])
    print(f"Similarity to '{sentence}'")
    print(f"Score: {similarity:.3f}\n")

##### Observations

Notice how:
- "A function in Python allows code reuse." has HIGH similarity (same meaning, different words)
- "Dogs are loyal animals." has LOW similarity (completely different)

**This is semantic search in action!**